# Detecting Topic Merges and Splits in Dynamic Political Conversations

Cláudia Oliveira

Supervisor - Prof. Dr. Álvaro Figueira

Faculty of Science, University of Porto

### Instalations

In [1]:
%%capture
!pip install  sentence-transformers  gensim scikit-learn pandas  tqdm emoji rapidfuzz nltk
import nltk
nltk.download('stopwords')
!pip install -U numpy==1.26.4
!pip install -U scipy==1.11.4
!pip install -U hdbscan==0.8.33
!pip install top2vec

### Importing necessary packages

In [1]:
from top2vec import Top2Vec
import pandas as pd
import numpy as np
from datetime import timedelta
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

In [2]:
from google.colab import files
uploaded = files.upload()

Saving 20news.csv to 20news.csv


### 119th Congress Tweets Dataset

In [ ]:
# ============================================================================================================================================================
# PARAMETERS
# ============================================================================================================================================================
window_size = timedelta(days=7)
step_size = timedelta(days=3)
top_n_words = 10
min_docs_per_topic = 50

# ============================================================================================================================================================
# LOAD & PREP
# ============================================================================================================================================================
tweets = pd.read_csv("cleanedtweets.csv", encoding="utf-8", low_memory=False)
tweets['created_at'] = pd.to_datetime(tweets['created_at'])
tweets = tweets.sort_values('created_at')

window_topics_t2v = []

start = tweets['created_at'].min()
current = start

# ============================================================================================================================================================
# CV + NPMI + UMASS + TQ + DIVERSITY
# ============================================================================================================================================================
def compute_tq_slice(topic_words_dict, docs, dictionary=None, top_n=10):
    topic_words = [words[:top_n] for words in topic_words_dict.values() if words]
    if len(topic_words) == 0:
        return None, None, None, None, None

    texts = [doc.split() for doc in docs]

    if dictionary is None:
        dictionary = Dictionary(texts)
        dictionary.filter_extremes(no_below=2, no_above=0.9)

    if len(dictionary) == 0:
        return None, None, None, None, None

    # ---- CV ----
    try:
        cm_cv = CoherenceModel(
            topics=topic_words, texts=texts,
            dictionary=dictionary, coherence='c_v'
        )
        cv_per_topic = cm_cv.get_coherence_per_topic()
        mean_cv = float(np.nanmean(cv_per_topic))
    except:
        cv_per_topic = []
        mean_cv = None

    # ---- NPMI ----
    try:
        cm_npmi = CoherenceModel(
            topics=topic_words, texts=texts,
            dictionary=dictionary, coherence='c_npmi'
        )
        npmi_per_topic = cm_npmi.get_coherence_per_topic()
        mean_npmi = float(np.nanmean(npmi_per_topic))
    except:
        npmi_per_topic = []
        mean_npmi = None

    # ---- UMass ----
    try:
        cm_umass = CoherenceModel(
            topics=topic_words, texts=texts,
            dictionary=dictionary, coherence='u_mass'
        )
        umass_per_topic = cm_umass.get_coherence_per_topic()
        mean_umass = float(np.nanmean(umass_per_topic))
    except:
        umass_per_topic = []
        mean_umass = None

    # ---- diversity ----
    K = len(topic_words)
    diversities = []
    for k, words_k in enumerate(topic_words):
        if len(words_k) == 0:
            continue
        overlaps = 0
        for i, words_j in enumerate(topic_words):
            if i == k:
                continue
            overlaps += len(set(words_k) & set(words_j)) / len(words_k)
        redundancy = overlaps / (K - 1) if K > 1 else 0.0
        diversities.append(1.0 - redundancy)

    mean_div = float(np.nanmean(diversities)) if diversities else None

    # ---- TQ ----
    if cv_per_topic and len(cv_per_topic) == len(diversities):
        tq = float(np.nanmean([c * d for c, d in zip(cv_per_topic, diversities)]))
    else:
        tq = None

    return mean_cv, mean_npmi, mean_umass, mean_div, tq


# ============================================================================================================================================================
# Window Look
# ============================================================================================================================================================
while current + window_size < tweets['created_at'].max():
    current_end = current + window_size
    subset = tweets[(tweets['created_at'] >= current) & (tweets['created_at'] < current_end)]

    if len(subset) < 10:
        current += step_size
        continue

    docs_window = subset["preprocess_top2vec"].tolist()

    top2vec_model = Top2Vec(
        documents=docs_window,
        speed="learn",
        embedding_model="all-MiniLM-L6-v2"
    )

    words_array, word_scores_array, topic_ids = top2vec_model.get_topics()

    topic_words = {}
    topic_doc_counts = {}

    for i, tid in enumerate(topic_ids):
        size = top2vec_model.topic_sizes.get(tid, 0)
        topic_doc_counts[tid] = size
        if size >= min_docs_per_topic:
            words = words_array[i][:top_n_words]
            if len(words) > 0:
                topic_words[tid] = list(words)

    print(f"Window {current.date()} -> {current_end.date()}: docs={len(subset)} "
          f"| all_topics={len(topic_ids)} | kept={len(topic_words)}")

    window_topics_t2v.append({
        "start_date": current,
        "end_date": current_end,
        "num_docs": len(subset),
        "topic_words": topic_words,
        "topic_doc_counts": topic_doc_counts,
        "tweets_with_topics": subset[["created_at", "preprocess_top2vec"]].to_dict(orient="records")
    })

    current += step_size


# ============================================================================================================================================================
# Metrics Per Window
# ============================================================================================================================================================
rows_t2v = []

for w in window_topics_t2v:
    window_docs = [d["preprocess_top2vec"] for d in w["tweets_with_topics"]]

    valid_topics = dict(sorted(
        ((tid, words) for tid, words in w["topic_words"].items() if len(words) > 0),
        key=lambda x: x[0]
    ))

    cv, npmi, umass, diversity, tq = compute_tq_slice(
        valid_topics,
        window_docs,
        dictionary=None,
        top_n=top_n_words
    )

    topic_strings = [" | ".join(words) for words in valid_topics.values()]
    topic_counts_list = [w["topic_doc_counts"].get(tid, 0) for tid in valid_topics.keys()]

    rows_t2v.append({
        "start_date": w["start_date"],
        "end_date": w["end_date"],
        "num_docs": w["num_docs"],
        "cv": cv,
        "npmi": npmi,
        "umass": umass,
        "diversity": diversity,
        "tq": tq,
        "topics": topic_strings,
        "topic_doc_counts": topic_counts_list
    })

df_t2v = pd.DataFrame(rows_t2v)

#### Saving Dataset

In [ ]:
df_t2v.to_csv("top2vecresults.csv", index=False)

### 20news Dataset

In [6]:
# ============================================================================================================
# 1. CONFIGURATION
# ============================================================================================================

min_docs_per_topic = 15
top_n_words = 10
embedding_model = "all-MiniLM-L6-v2"

# ============================================================================================================
# 2. LOAD DATA
# ============================================================================================================

import pandas as pd
import numpy as np
import ast
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from top2vec import Top2Vec

df = pd.read_csv("20news.csv", encoding="utf-8", low_memory=False)

docs = df["top2vec_text"].tolist()

# ============================================================================================================
# 3. RUN TOP2VEC
# ============================================================================================================

t2v_model = Top2Vec(
    documents=docs,
    speed="learn",
    embedding_model=embedding_model
)

topic_words_array, scores_array, topic_ids = t2v_model.get_topics()

topic_sizes = t2v_model.topic_sizes
topic_words = {}
topic_doc_counts = {}

for i, tid in enumerate(topic_ids):
    size = topic_sizes.get(tid, 0)
    topic_doc_counts[tid] = size

    if size >= min_docs_per_topic:
        words = topic_words_array[i][:top_n_words]
        topic_words[tid] = list(words)

df["topic_id"] = t2v_model.get_documents_topics(list(range(len(docs))))[0]

print(f"Extracted {len(topic_words)} topics.")


2025-11-26 11:28:18,332 - top2vec - INFO - Pre-processing documents for training
INFO:top2vec:Pre-processing documents for training
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2025-11-26 11:28:29,028 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model
INFO:top2vec:Downloading all-MiniLM-L6-v2 model
2025-11-26 11:28:30,374 - top2vec - INFO - Creating joint document/word embedding
INFO:top2vec:Creating joint document/word embedding
2025-11-26 11:34:44,894 - top2vec - INFO - Creating lower dimension embedding of documents
INFO:top2vec:Creating lower dimension embedding of documents
2025-11-26 11:34:56,616 - top2vec - INFO - Finding dense areas of documents
INFO:top2vec:Finding dense areas of documents
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 

Extracted 20 topics.


In [7]:
# ============================================================================================================
# 4. Metrics: CV, NPMI, UMass, Diversity, TQ
# ============================================================================================================

def compute_tq(topic_words_dict, docs, dictionary=None, top_n=10):

    topic_words = [words[:top_n] for words in topic_words_dict.values() if words]
    if not topic_words:
        return None, None, None, None, None

    texts = [doc.split() for doc in docs]

    if dictionary is None:
        dictionary = Dictionary(texts)
        dictionary.filter_extremes(no_below=5, no_above=0.9)

    if len(dictionary) == 0:
        return None, None, None, None, None

    # Coherence CV
    cm_cv = CoherenceModel(
        topics=topic_words,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    cv_scores = cm_cv.get_coherence_per_topic()

    # Coherence NPMI
    cm_npmi = CoherenceModel(
        topics=topic_words,
        texts=texts,
        dictionary=dictionary,
        coherence='c_npmi'
    )
    npmi_scores = cm_npmi.get_coherence_per_topic()

    # Coherence UMass
    cm_umass = CoherenceModel(
        topics=topic_words,
        texts=texts,
        dictionary=dictionary,
        coherence='u_mass'
    )
    umass_scores = cm_umass.get_coherence_per_topic()

    # Diversity
    K = len(topic_words)
    diversities = []

    for k, words_k in enumerate(topic_words):
        overlaps = 0
        for i, words_j in enumerate(topic_words):
            if i == k:
                continue
            overlaps += len(set(words_k) & set(words_j)) / len(words_k)
        redundancy = overlaps / (K - 1) if K > 1 else 0
        diversities.append(1 - redundancy)

    # TQ
    tq = np.mean([c * d for c, d in zip(cv_scores, diversities)])

    return (
        tq,
        np.mean(cv_scores),
        np.mean(diversities),
        np.mean(npmi_scores),
        np.mean(umass_scores)
    )

dictionary = Dictionary([doc.split() for doc in docs])

tq, cv_mean, div_mean, npmi_mean, umass_mean = compute_tq(
    topic_words,
    docs,
    dictionary,
    top_n_words
)

print("\n===============================")
print(" TOP2VEC TOPIC QUALITY METRICS")
print("===============================")
print("TQ:", tq)
print("CV:", cv_mean)
print("Diversity:", div_mean)
print("NPMI:", npmi_mean)
print("UMass:", umass_mean)


# ============================================================================================================
# 5. PURITY
# ============================================================================================================

from sklearn.metrics import confusion_matrix

def compute_topic_purity(df, topic_column="topic_id", label_column="label"):

    df_valid = df.copy()

    topics = df_valid[topic_column].astype(int).values
    labels = df_valid[label_column].astype(str).values

    unique_labels = sorted(df_valid[label_column].unique())
    label_to_id = {lbl: i for i, lbl in enumerate(unique_labels)}
    y_true = np.array([label_to_id[l] for l in labels])

    unique_topics = sorted(df_valid[topic_column].unique())
    topic_to_id = {t: i for i, t in enumerate(unique_topics)}
    y_pred = np.array([topic_to_id[t] for t in topics])

    cm = confusion_matrix(y_pred, y_true)
    purity = cm.max(axis=1).sum() / cm.sum()

    return purity, cm, unique_topics, unique_labels


purity, cm, unique_topics, unique_labels = compute_topic_purity(df)

print("\n===============================")
print(" TOPIC PURITY (TOP2VEC)")
print("===============================")
print(f"Purity: {purity:.4f}")
print("Clusters:", len(unique_topics))
print("True labels:", len(unique_labels))

print("\nDominant category per topic:")
for i, topic_id in enumerate(unique_topics):
    dom = cm[i].argmax()
    print(f"Topic {topic_id} → {unique_labels[dom]} (count={cm[i].max()})")


 TOP2VEC TOPIC QUALITY METRICS
TQ: 0.4079609007024122
CV: 0.4175933374701984
Diversity: 0.9752631578947367
NPMI: -0.233447845670602
UMass: -4.685224685001423

 TOPIC PURITY (TOP2VEC)
Purity: 0.8141
Clusters: 20
True labels: 3

Dominant category per topic:
Topic 0 → talk.politics.mideast (count=476)
Topic 1 → talk.politics.guns (count=363)
Topic 2 → talk.politics.mideast (count=221)
Topic 3 → talk.politics.guns (count=82)
Topic 4 → talk.politics.misc (count=165)
Topic 5 → talk.politics.misc (count=56)
Topic 6 → talk.politics.guns (count=72)
Topic 7 → talk.politics.guns (count=68)
Topic 8 → talk.politics.misc (count=71)
Topic 9 → talk.politics.guns (count=90)
Topic 10 → talk.politics.misc (count=65)
Topic 11 → talk.politics.mideast (count=87)
Topic 12 → talk.politics.misc (count=78)
Topic 13 → talk.politics.mideast (count=31)
Topic 14 → talk.politics.misc (count=43)
Topic 15 → talk.politics.misc (count=38)
Topic 16 → talk.politics.misc (count=40)
Topic 17 → talk.politics.misc (count=33)